In [1]:
!pip install -q transformers sentence-transformers razdel sacremoses bert_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.6 MB/s eta 0:00:00


# Генерация train_backtranslate.csv

In [2]:
import re
import random
import warnings
import time

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

warnings.filterwarnings('ignore')

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)

Device: cuda


In [4]:
from google.colab import drive
import os

drive.mount('/content/drive')
drive_root = '/content/drive/MyDrive/papadyk-collab/vkr'
output_dir = os.path.join(drive_root, 'output')
os.makedirs(output_dir, exist_ok=True)

train_path = os.path.join(drive_root, 'train.csv')
out_path = os.path.join(output_dir, 'backtranslate-3')

Mounted at /content/drive


In [5]:
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

from huggingface_hub import login
login(HF_TOKEN)

In [6]:
TARGET_PER_CLASS = 40
SIM_MIN = 0.85
SIM_MAX = 0.95
SIM_LABEL_MIN = 0.8
SIM_LABEL_MAX = 0.99

In [7]:
df = pd.read_csv(train_path)
counts = df["label"].value_counts()
small_labels = counts[counts < TARGET_PER_CLASS].sort_values(ascending=True).index
df_small = df[df["label"].isin(small_labels)]

# Расчёт общего числа аугментаций
TOTAL_AUGMENTED = (TARGET_PER_CLASS * len(small_labels)) - len(df_small)

print(f"Всего малых классов: {len(small_labels)}")
print(f"Текущее количество примеров в малых классах: {len(df_small)}")
print(f"Целевое количество на класс: {TARGET_PER_CLASS}")
print(f"Нужно сгенерировать аугментаций: {TOTAL_AUGMENTED}")

Всего малых классов: 26
Текущее количество примеров в малых классах: 367
Целевое количество на класс: 40
Нужно сгенерировать аугментаций: 673


In [8]:
source_text = "[ORGANIZATION] Почтовый/Юридический адрес: [LOCATION] Телефон: [CONTACT]; [CONTACT], факс: [CONTACT] e-mail: [CONTACT] ОКПО [ID], ОГРН [ID], ИНН/КПП [ID]/[ID] от [DATE_TIME] No [DOCUMENT_NUMBER] [DOCUMENT_NUMBER] [DATE_TIME] г. на No от Руководителю проекта [OBJECT] [PERSON] [CONTACT] О проезде техники через КПП [OBJECT] Уважаемый [PERSON]! На письмо исх. No [DOCUMENT_NUMBER] от [DATE_TIME] сообщаем Вам, что [ORGANIZATION] не согласовывает проезд транспорта [ORGANIZATION] через ДКП-1 с проездом по территории [OBJECT] и выездом через ДКП-2 ([NUMBER] км) к трассе трубопровода [ORGANIZATION]. Рекомендуем Вам рассмотреть проезд по вдольтрассовому проезду (ВТП) [ORGANIZATION] с заездом через ДКП-3 ([NUMBER] км), а/т должен быть оснащен искрогасителями, также персонал иметь при себе подтверждающий документ о прохождении инструктажа по требованиям пожарной безопасности. В случае Вашего согласия просим подтвердить официальным письмом. Обращаем Ваше внимание, что в мае-июне на ВТП [ORGANIZATION] будут действовать сезонные ограничения по проезду техники, связанные с паводковым периодом в [LOCATION], просим Вас заблаговременно уточнять в [ORGANIZATION] о возможности проезда по ВТП. Для оформления пропусков просим Вас направить [PERSON], [PERSON], наименование перевозимого груза с привязкой к автомобильному транспорту в форме заявки (приложение), а также лист ознакомления со Стандартом [ORGANIZATION] пропускной и внутритобъектовый режимы на территории производственных и иных объектов No [DOCUMENT_NUMBER]. Приложение: Образец заявки на проезд – на 1 л. в 1 экз. Первый заместитель генерального директора по производству – главный инженер [PERSON] [PERSON] [CONTACT], доб. [NUMBER] [CONTACT] ДОКУМЕНТ ПОДЛИННАЯ ЭЛЕКТРОННОЙ ПОДЛИННОСТИ Сертификат [FINANCIAL_DATA] [FINANCIAL_DATA] Владелец [PERSON] Действителен с [DATE_TIME] по [DATE_TIME]"

In [9]:
from sentence_transformers import SentenceTransformer, util

# -------- Модель эмбеддингов --------
embed_model_name = "deepvk/USER2-base"
embed_model = SentenceTransformer(embed_model_name, device=device)

def cos_sim(model, text1, text2):
  emb1 = model.encode(text1, convert_to_tensor=True, normalize_embeddings=True)
  emb2 = model.encode(text2, convert_to_tensor=True, normalize_embeddings=True)
  return util.cos_sim(emb1, emb2).item()

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/359 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/837 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [10]:
from transformers import MarianMTModel, MarianTokenizer

# -------- Модели переводов --------
ru_en_name = "Helsinki-NLP/opus-mt-ru-en"
en_ru_name = "Helsinki-NLP/opus-mt-en-ru"

ru_fr_name = "Helsinki-NLP/opus-mt-ru-fr"
fr_ru_name = "Helsinki-NLP/opus-mt-fr-ru"

ru_es_name = "Helsinki-NLP/opus-mt-ru-es"
es_ru_name = "Helsinki-NLP/opus-mt-es-ru"

ru_en_tokenizer = MarianTokenizer.from_pretrained(ru_en_name)
ru_en_model = MarianMTModel.from_pretrained(ru_en_name).to(device)

en_ru_tokenizer = MarianTokenizer.from_pretrained(en_ru_name)
en_ru_model = MarianMTModel.from_pretrained(en_ru_name).to(device)

ru_fr_tokenizer = MarianTokenizer.from_pretrained(ru_fr_name)
ru_fr_model = MarianMTModel.from_pretrained(ru_fr_name).to(device)

fr_ru_tokenizer = MarianTokenizer.from_pretrained(fr_ru_name)
fr_ru_model = MarianMTModel.from_pretrained(fr_ru_name).to(device)

ru_es_tokenizer = MarianTokenizer.from_pretrained(ru_es_name)
ru_es_model = MarianMTModel.from_pretrained(ru_es_name).to(device)

es_ru_tokenizer = MarianTokenizer.from_pretrained(es_ru_name)
es_ru_model = MarianMTModel.from_pretrained(es_ru_name).to(device)

def get_model_by_mode(mode: str):
    if mode == "ru-en":
        tokenizer, model = ru_en_tokenizer, ru_en_model
    elif mode == "en-ru":
        tokenizer, model = en_ru_tokenizer, en_ru_model
    elif mode == "ru-fr":
        tokenizer, model = ru_fr_tokenizer, ru_fr_model
    elif mode == "fr-ru":
        tokenizer, model = fr_ru_tokenizer, fr_ru_model
    elif mode == "ru-es":
        tokenizer, model = ru_es_tokenizer, ru_es_model
    elif mode == "es-ru":
        tokenizer, model = es_ru_tokenizer, es_ru_model
    else:
        raise ValueError(f"Unknown mode: {mode}")
    return tokenizer, model

def mt_tokens_len(text: str, mode: str) -> int:
    tokenizer, model = get_model_by_mode(mode)
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=False,
        padding=False,
    )
    return inputs.input_ids.shape[1]


def generate_translate(
    text: str,
    mode: str,
    **gen_kwargs
) -> str:
    tokenizer, model = get_model_by_mode(mode)

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=False,
        padding=True,
    ).to(device)

    input_len = inputs.input_ids.shape[1]
    if input_len > 300:
        raise ValueError(
            f"Input text too long for translation: {input_len} tokens (max 300). "
            f"Use split_long_sentence() before calling generate_translate()."
        )

    # дефолтные параметры генерации
    default_gen_kwargs = dict(
        num_beams=1,
        do_sample=True,
        top_k=50,
        top_p=0.92,
        temperature=1.1,
    )
    default_gen_kwargs.update(gen_kwargs)

    with torch.no_grad():
        generated_tokens = model.generate(
            **inputs,
            **default_gen_kwargs
        )

    return tokenizer.decode(generated_tokens[0], skip_special_tokens=True)

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/803k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/309M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

In [11]:
# -------- Модель перплексии --------
from transformers import AutoTokenizer, AutoModelForCausalLM

RUGPT_MODEL_NAME = "sberbank-ai/rugpt3small_based_on_gpt2"
rugpt_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

rugpt_tok = AutoTokenizer.from_pretrained(RUGPT_MODEL_NAME)
rugpt_model = AutoModelForCausalLM.from_pretrained(RUGPT_MODEL_NAME).to(rugpt_device)
rugpt_model.eval()


@torch.no_grad()
def rugpt_perplexity_list(texts: list[str], max_length: int = 512, batch_size: int = 8) -> list[float]:
    """
    Возвращает список перплексий для списка текстов с батч-обработкой.
    Perplexity считается как exp(средний loss по токенам).
    """
    if not texts:
        return []

    ppl_values = []

    # Обработка батчами
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]

        # Токенизация батча
        enc = rugpt_tok(
            batch_texts,
            return_tensors="pt",
            truncation=True,
            max_length=max_length,
            padding=True  # ВАЖНО для батчинга
        )
        input_ids = enc["input_ids"].to(rugpt_device)
        attention_mask = enc["attention_mask"].to(rugpt_device)

        # Прогон батча
        outputs = rugpt_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=input_ids
        )

        # Расчёт perplexity для каждого текста в батче
        # outputs.loss — это средний loss по всему батчу
        # Нужно считать loss для каждого примера отдельно
        logits = outputs.logits  # shape: (batch_size, seq_len, vocab_size)

        for j in range(len(batch_texts)):
            # Получаем токены и маску для текущего примера
            tokens = input_ids[j]
            mask = attention_mask[j]

            # Считаем loss только по реальным токенам (не паддингу)
            seq_len = mask.sum().item()

            if seq_len == 0:
                ppl_values.append(float("inf"))
                continue

            # Сдвиг для language modeling: предсказываем следующий токен
            shift_logits = logits[j, :-1, :]  # (seq_len-1, vocab_size)
            shift_labels = tokens[1:]  # (seq_len-1,)
            shift_mask = mask[1:]  # (seq_len-1,)

            # Считаем cross-entropy loss
            loss_fct = torch.nn.CrossEntropyLoss(reduction='none')
            loss_per_token = loss_fct(shift_logits, shift_labels)  # (seq_len-1,)

            # Усредняем только по реальным токенам
            masked_loss = loss_per_token * shift_mask
            avg_loss = masked_loss.sum() / shift_mask.sum()

            ppl = torch.exp(avg_loss).item()
            ppl_values.append(ppl)

    return ppl_values

model.safetensors:   0%|          | 0.00/309M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/720 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/574 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/551M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: sberbank-ai/rugpt3small_based_on_gpt2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 
transformer.h.{0...11}.attn.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [18]:
import torch
import re
from razdel import sentenize
from tqdm.auto import tqdm
from sentence_transformers import util

# ---------- ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ ----------

def clean_bt_result(text: str) -> str:
    text = re.sub(r'([,.!?])\1{2,}', r'\1', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\s+,', ',', text)
    text = re.sub(r',\s*', ', ', text)
    return text.strip()


def mask_placeholders(text: str) -> tuple[str, dict]:
    """
    Заменяем [ORGANIZATION], [DATE_TIME] и т.п. на MASK_0, MASK_1, ...
    чтобы плейсхолдеры не портились при переводе.
    """
    pattern = r'\[[^\]]+\]'
    mapping = {}
    counter = [0]

    def replacer(m):
        token = m.group(0)
        key = f"MASK_{counter[0]}"
        mapping[key] = token
        counter[0] += 1
        return key

    masked = re.sub(pattern, replacer, text)
    return masked, mapping


def unmask_placeholders(text: str, mapping: dict) -> str:
    for key, original in mapping.items():
        text = text.replace(key, original)
    return text

def preprocess_before_translate(text: str) -> str:
    # Убираем строки из повторяющихся спецсимволов (таблицы, разделители)
    text = re.sub(r'[-=_*]{5,}', '—', text)   # 5+ дефисов/равно/подчёркиваний → тире
    text = re.sub(r'[.\s]{5,}', ' ', text)     # 5+ точек с пробелами → пробел
    text = re.sub(r'\s{2,}', ' ', text)        # множественные пробелы
    return text.strip()

def split_long_sentence(sent: str, max_tokens: int, mode: str) -> list[str]:
    if mt_tokens_len(sent, mode=mode) <= max_tokens:
        return [sent]

    parts = re.split(r'(?<=[,;:—\-])\s+', sent)

    chunks = []

    if len(parts) == 1:
        # нет разделителей — режем по словам
        words = sent.split()
        current = ""
        for w in words:
            candidate = f"{current} {w}".strip() if current else w
            if mt_tokens_len(candidate, mode=mode) <= max_tokens:
                current = candidate
            else:
                if current:
                    chunks.append(current)
                current = w
        if current:
            chunks.append(current)
        return chunks if chunks else [sent]

    # есть разделители — режем по ним, рекурсивно обрабатываем каждую часть
    current = ""
    for part in parts:
        candidate = f"{current} {part}".strip() if current else part
        if mt_tokens_len(candidate, mode=mode) <= max_tokens:
            current = candidate
        else:
            if current:
                # ✅ рекурсия: если текущий накопленный чанк вдруг > max_tokens
                chunks.extend(split_long_sentence(current, max_tokens, mode))
            # ✅ рекурсия: обрабатываем новую часть отдельно
            if mt_tokens_len(part, mode=mode) > max_tokens:
                chunks.extend(split_long_sentence(part, max_tokens, mode))
                current = ""
            else:
                current = part
    if current:
        chunks.extend(split_long_sentence(current, max_tokens, mode))

    return chunks if chunks else [sent]


def safe_translate(text: str, mode: str, max_tokens: int, **gen_kwargs) -> str:
    text = preprocess_before_translate(text)   # ← добавить сюда
    chunks = split_long_sentence(text, max_tokens=max_tokens, mode=mode)
    out_chunks = []
    for ch in chunks:
        ch_len = mt_tokens_len(ch, mode=mode)
        if ch_len > 300:
            words = ch.split()
            for i in range(0, len(words), 20):
                sub = " ".join(words[i:i+20])
                if sub.strip():
                    out_chunks.append(generate_translate(sub, mode=mode, **gen_kwargs))
        else:
            out_chunks.append(generate_translate(ch, mode=mode, **gen_kwargs))
    return " ".join(out_chunks)

def generate_bt_candidates_for_sentence(text: str):
    candidates = []

    # Режимы генерации
    modes = [
        {"do_sample": False, "num_beams": 5},
        {"do_sample": True, "top_p": 0.90, "temperature": 1.0},
        {"do_sample": True, "top_p": 0.95, "temperature": 1.2},
    ]

    # ru -> en -> ru
    for cfg in modes:
        # 1) ru -> en (режем под ru-en)
        en = safe_translate(
            text,
            mode="ru-en",
            max_tokens=120,   # как в split_long_sentence при back_translate_document
            **cfg
        )

        # 2) en -> ru (режем под en-ru, но уже до 300)
        ru_bt = safe_translate(
            en,
            mode="en-ru",
            max_tokens=300,
            **cfg
        )

        candidates.append(clean_bt_result(ru_bt))

    # ru -> fr -> ru
    for cfg in modes:
        fr = safe_translate(
            text,
            mode="ru-fr",
            max_tokens=120,
            **cfg
        )

        ru_bt = safe_translate(
            fr,
            mode="fr-ru",
            max_tokens=300,
            **cfg
        )

        candidates.append(clean_bt_result(ru_bt))

    # ru -> es -> ru
    for cfg in modes:
        es = safe_translate(
            text,
            mode="ru-es",
            max_tokens=120,
            **cfg
        )

        ru_bt = safe_translate(
            es,
            mode="es-ru",
            max_tokens=300,
            **cfg
        )

        candidates.append(clean_bt_result(ru_bt))

    # Убираем дубликаты
    candidates = list(dict.fromkeys([c.strip() for c in candidates if c.strip()]))
    return candidates

def choose_best_bt_for_sentence(source_chunk: str, candidates: list[str]) -> tuple[str, float]:
    """
    Выбираем лучший BT-кандидат для одного куска:
    - считаем cosine similarity со source_chunk,
    - фильтруем по [sim_min, sim_max],
    - для прошедших считаем перплексию ruGPT,
    - выбираем кандидата с минимальной перплексией,
      при равенстве – с максимальным cosine.
    """
    if not candidates:
        return source_chunk, 1.0

    # 1. эмбеддинги source + candidates
    texts = [source_chunk] + candidates
    embs = embed_model.encode(
        texts,
        convert_to_tensor=True,
        normalize_embeddings=True
    )

    src_emb = embs[0]
    cand_embs = embs[1:]
    sims = util.cos_sim(src_emb, cand_embs)[0]  # shape: (N,)

    # 2. фильтр по cosine
    mask = (sims >= SIM_MIN) & (sims <= SIM_MAX)

    if mask.any():
        idxs = torch.nonzero(mask, as_tuple=False).squeeze(1)
        filtered_candidates = [candidates[i] for i in idxs.tolist()]
        filtered_sims = sims[idxs]

        # 3. считаем перплексию для прошедших кандидатов
        ppls = rugpt_perplexity_list(filtered_candidates, batch_size=8)

        # 4. выбираем по минимальной перплексии, при равенстве – по максимальному cosine
        best_idx_local = min(
            range(len(filtered_candidates)),
            key=lambda i: (ppls[i], -filtered_sims[i].item())
        )

        best_text = filtered_candidates[best_idx_local]
        best_sim = filtered_sims[best_idx_local].item()

    else:
        # никто не попал в окно по cosine – fallback: берём максимум cosine
        best_idx = torch.argmax(sims).item()
        best_text = candidates[best_idx]
        best_sim = sims[best_idx].item()

    return best_text, best_sim

def normalize_text_for_compare(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s]', '', text)  # убираем пунктуацию
    return text

def back_translate_document(text_orig):
    masked_text, mapping = mask_placeholders(text_orig)
    masked_text = preprocess_before_translate(masked_text)

    sentences = [s.text.strip() for s in sentenize(masked_text) if s.text.strip()]

    bt_sentences = []

    for s in tqdm(sentences, total=len(sentences), desc="Sentences"):
        cand_list = generate_bt_candidates_for_sentence(s)
        best_text, best_sim = choose_best_bt_for_sentence(s, cand_list)
        bt_sentences.append(best_text)

    result_masked = " ".join(bt_sentences)
    bt_text = unmask_placeholders(result_masked, mapping)

    return bt_text

In [19]:
bt_text = back_translate_document(source_text)
cos_score = cos_sim(embed_model, source_text, bt_text)

print(f"\n=== BACK-TRANSLATION ===")
print(f"Результат: {len(source_text)} -> {len(bt_text)} символов")

print(f"\nCosine similarity (оригинал ↔ BT): {cos_score:.3f}")

Sentences:   0%|          | 0/11 [00:00<?, ?it/s]


=== BACK-TRANSLATION ===
Результат: 1481 -> 941 символов

Cosine similarity (оригинал ↔ BT): 0.861


In [20]:
import copy
import random
import os
import pandas as pd
import torch
from tqdm.auto import tqdm
from sentence_transformers import util

aug_rows = []

aug_file_path = os.path.join(out_path, "train_backtranslate_partial.csv")
if os.path.exists(aug_file_path):
    df_prev = pd.read_csv(aug_file_path)
    aug_rows = df_prev.to_dict(orient="records")
    print(f"Загружено уже сгенерированных примеров: {len(aug_rows)}")
else:
    print("Генерируем с нуля")

label_to_texts = {
    label: df_small.loc[df_small["label"] == label, "text"].tolist()
    for label in small_labels
}

label_to_embs = {}

for label in tqdm(small_labels, desc="Labels"):
    texts_orig = label_to_texts[label]
    orig_count = len(texts_orig)

    label_aug_rows = [r for r in aug_rows if r["label"] == label]
    label_aug_rows_count = len(label_aug_rows)
    current_count_with_aug = orig_count + label_aug_rows_count
    need = TARGET_PER_CLASS - current_count_with_aug

    print(f"\nLabel: {label} | есть {current_count_with_aug}, нужно добить: {need}")

    if need <= 0:
        print(f"Лейбл {label} уже заполнен")
        continue

    # эмбеддинги всех оригинальных текстов этого лейбла
    label_embs = label_to_embs.get(label)
    if label_embs is None or label_embs.numel() == 0:
        texts_label = copy.deepcopy(texts_orig)
        # for row in aug_rows:
        #     if row["label"] == label:
        #         texts_label.append(row["text"])

        label_embs = embed_model.encode(
            texts_label,
            convert_to_tensor=True,
            normalize_embeddings=True
        )
        label_to_embs[label] = label_embs

    orig_idx = 0
    attempts = 0
    max_attempts = need * 20

    while need > 0 and attempts < max_attempts:
        attempts += 1

        source_text = texts_orig[orig_idx]

        bt_text = back_translate_document(source_text)

        if normalize_text_for_compare(bt_text) == normalize_text_for_compare(source_text):
          print(f"[SKIP] BT идентичен оригиналу: '{source_text[:60]}...'")
          orig_idx = (orig_idx + 1) % orig_count
          continue

        original_cosine_sim = cos_sim(embed_model, source_text, bt_text)
        # 1. схожесть с исходным текстом
        if not (SIM_MIN <= original_cosine_sim <= SIM_MAX):
            print(
                f"Пропуск: сходство с source_text не прошло | "
                f"original_cosine_sim={original_cosine_sim:.4f}, "
                f"ожидалось [{SIM_MIN}, {SIM_MAX}]"
            )
            orig_idx = (orig_idx + 1) % orig_count
            continue

        # 2. схожесть с текстами этого же лейбла — проверяем ТОЛЬКО НА ФИНАЛЬНОМ BT-ТЕКСТЕ
        new_emb = embed_model.encode(
            bt_text,
            convert_to_tensor=True,
            normalize_embeddings=True
        )

        sims = util.cos_sim(new_emb, label_embs)[0]
        max_label_cosine_sim = float(torch.max(sims))

        if not (SIM_LABEL_MIN <= max_label_cosine_sim <= SIM_LABEL_MAX):
            print(
                f"Пропуск: сходство с текстами лейбла не прошло | "
                f"max_label_cos={max_label_cosine_sim:.4f}, "
                f"ожидалось [{SIM_LABEL_MIN}, {SIM_LABEL_MAX}]"
            )
            orig_idx = (orig_idx + 1) % orig_count
            continue

        # 3. добавляем в корпус лейбла и его эмбеддинги
        label_embs = torch.cat([label_embs, new_emb.unsqueeze(0)], dim=0)
        label_to_embs[label] = label_embs

        aug_rows.append({
            "label": label,
            "text": bt_text,
            "source_text": source_text,
            "cosine_sim": original_cosine_sim,
            "max_label_cosine_sim": max_label_cosine_sim,
            "augmentation_type": "back_translation",
        })

        df_aug_partial = pd.DataFrame(aug_rows)
        df_aug_partial.to_csv(aug_file_path, index=False)
        print(f"Сохранено {len(aug_rows)} аугментированных примеров в {aug_file_path}")

        orig_idx = (orig_idx + 1) % orig_count
        need -= 1

# финальное сохранение аугментаций
df_aug = pd.DataFrame(aug_rows)
df_aug.to_csv(aug_file_path, index=False)
print(f"\nИтого аугментированных примеров: {len(df_aug)}")
print(f"Итоговый файл с аугментацией сохранён в: {aug_file_path}")

# склейка с исходным df
df_full = pd.concat([df, df_aug[["label", "text"]]], ignore_index=True)
final_path = os.path.join(out_path, "train_backtranslate.csv")
df_full.to_csv(final_path, index=False)
print(f"Финальный тренировочный датасет сохранён в: {final_path}")

Загружено уже сгенерированных примеров: 524


Labels:   0%|          | 0/26 [00:00<?, ?it/s]


Label: Проект «Трубопроводный транспорт Ещё одного НГКМ» | есть 40, нужно добить: 0
Лейбл Проект «Трубопроводный транспорт Ещё одного НГКМ» уже заполнен

Label: Блок заместителя генерального директора по строительству | есть 40, нужно добить: 0
Лейбл Блок заместителя генерального директора по строительству уже заполнен

Label: Имущественные вопросы | есть 40, нужно добить: 0
Лейбл Имущественные вопросы уже заполнен

Label: Подразделение по информационным технологиям | есть 40, нужно добить: 0
Лейбл Подразделение по информационным технологиям уже заполнен

Label: Проект «Обустройство объектов Новейшей нейти» | есть 40, нужно добить: 0
Лейбл Проект «Обустройство объектов Новейшей нейти» уже заполнен

Label: Блок исполнительного директора по реализации проекта "Большое месторождение" | есть 40, нужно добить: 0
Лейбл Блок исполнительного директора по реализации проекта "Большое месторождение" уже заполнен

Label: Проект "Обустройство площадных объектов НГКМ Поменбше" | есть 40, нужно доби

Sentences:   0%|          | 0/5 [00:00<?, ?it/s]

Сохранено 525 аугментированных примеров в /content/drive/MyDrive/papadyk-collab/vkr/output/backtranslate-3/train_backtranslate_partial.csv


...

Сохранено 671 аугментированных примеров в /content/drive/MyDrive/papadyk-collab/vkr/output/backtranslate-3/train_backtranslate_partial.csv

Label: Проект сервиса скважин | есть 38, нужно добить: 2


Sentences:   0%|          | 0/4 [00:00<?, ?it/s]

Сохранено 672 аугментированных примеров в /content/drive/MyDrive/papadyk-collab/vkr/output/backtranslate-3/train_backtranslate_partial.csv


Sentences:   0%|          | 0/4 [00:00<?, ?it/s]

Сохранено 673 аугментированных примеров в /content/drive/MyDrive/papadyk-collab/vkr/output/backtranslate-3/train_backtranslate_partial.csv

Итого аугментированных примеров: 673
Итоговый файл с аугментацией сохранён в: /content/drive/MyDrive/papadyk-collab/vkr/output/backtranslate-3/train_backtranslate_partial.csv
Финальный тренировочный датасет сохранён в: /content/drive/MyDrive/papadyk-collab/vkr/output/backtranslate-3/train_backtranslate.csv
